# 02 — Pré-processamento

Limpeza, tratamento de ausentes, encoding e normalização/padronização.

In [1]:
# Imports
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
%load_ext autoreload
%autoreload 2
from utils import load_parquet_safe

df_path = "../data/df_model_raw.parquet"

df = load_parquet_safe(df_path, '01_eda.ipynb')

print("Dados carregados com sucesso!")

Dados carregados com sucesso!


In [3]:
df.shape

(717932, 22)

In [4]:
df

,SEMAGESTAC,GESTACAO,IDADEMAE,ESCMAE2010,ESTCIVMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,GRAVIDEZ,MESPRENAT,CONSPRENAT,SEXO,KOTELCHUCK,IDADEPAI,LATITUDE,LONGITUDE,PREMATURO,PAI_AUSENTE,IDADEPAI_INVALIDA
0,39.0,5.0,29,5.0,1.0,1,0.0,0.0,0.0,0.0,0.0,1.0,3.0,13.0,1,5,NaN,-21.6088,-45.5691,0,1,0
1,38.0,5.0,43,5.0,2.0,1,4.0,0.0,2.0,2.0,2.0,1.0,3.0,8.0,1,5,NaN,-21.8360,-45.4004,0,1,0
2,39.0,5.0,33,5.0,1.0,1,1.0,0.0,1.0,1.0,0.0,1.0,2.0,11.0,1,5,NaN,-21.5556,-45.4364,0,1,0
3,38.0,5.0,26,2.0,1.0,2,4.0,0.0,4.0,4.0,0.0,1.0,5.0,4.0,2,2,NaN,-21.5556,-45.4364,0,1,0
4,37.0,5.0,20,4.0,1.0,1,0.0,0.0,0.0,0.0,0.0,1.0,4.0,7.0,1,2,NaN,-21.7579,-45.5391,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
717927,35.0,4.0,37,3.0,2.0,4,2.0,1.0,1.0,2.0,0.0,1.0,2.0,7.0,2,5,35.0,-19.9102,-43.9266,1,0,0
717928,38.0,5.0,33,3.0,1.0,2,1.0,1.0,0.0,1.0,0.0,1.0,4.0,11.0,2,2,23.0,-19.9102,-43.9266,0,0,0
717929,38.0,5.0,30,3.0,2.0,1,1.0,1.0,0.0,1.0,0.0,1.0,2.0,8.0,2,5,32.0,-19.9102,-43.9266,0,0,0
717930,40.0,5.0,28,3.0,1.0,4,0.0,0.0,0.0,0.0,0.0,1.0,2.0,10.0,2,5,39.0,-19.9102,-43.9266,0,0,0


## Convertendo Sentinelas (Sentinelas -> NaNs)
O SINASC tem sentinelas que são ignorados pelo sistema, porém contentdo 9,99,999 como códigos a serem ignorados.

Objetivo: Equalizar os dados para que o Pandas consiga comparar as linhas de forma justa na etapa de duplicadas

### Como identificar as colunas que tem sentinela?

#### 1. Técnica da Frequência de Finais
Ao fazer um `value_counts().head(10)`conseguimos o Top 10 valores mais populares por coluna

In [5]:
cols_to_check = df.columns

for col in cols_to_check:
    top_values = df[col].value_counts(normalize=True).head(10)
    print(f"Coluna: {col}")
    print(top_values)
    print("-" * 30)

Coluna: SEMAGESTAC
SEMAGESTAC
39.0    0.292670
38.0    0.231438
40.0    0.173330
37.0    0.109494
41.0    0.063487
36.0    0.043616
35.0    0.022797
34.0    0.015442
42.0    0.010665
33.0    0.008632
Name: proportion, dtype: float64
------------------------------
Coluna: GESTACAO
GESTACAO
5.0    0.870418
4.0    0.096395
6.0    0.016582
3.0    0.010546
2.0    0.005475
1.0    0.000584
Name: proportion, dtype: float64
------------------------------
Coluna: IDADEMAE
IDADEMAE
26    0.050069
25    0.049782
27    0.049726
28    0.049204
30    0.048591
29    0.048590
24    0.048392
31    0.047537
23    0.046957
22    0.046346
Name: proportion, dtype: float64
------------------------------
Coluna: ESCMAE2010
ESCMAE2010
3.0    0.557242
5.0    0.202487
2.0    0.167648
4.0    0.049514
1.0    0.020321
9.0    0.001443
0.0    0.001345
Name: proportion, dtype: float64
------------------------------
Coluna: ESTCIVMAE
ESTCIVMAE
1.0    0.452909
2.0    0.419698
5.0    0.101127
4.0    0.021165
9.0    0.002

### Como identificar o que é "lixo" do que é dado real? Será que todos os 9 na tabela toda são códigos de "ignorado"?

#### 2. Três critérios: Frequência, semântica e descontinuidade

Às vezes pelo nome da tabela essa informação consegue ser óbvia, mas outra vezes conseguimos seguir pelo o que os dados estão dizendo.

1. Onde o 9 e 99 são claramente SENTINELAS (lixo)
    - `MESPRENAT` **(99.0 com 2%)**: É sentinela. Ninguém começa o pré-natal no 99º mês. Note que o 9 aqui não aparece, o erro padrão do SINASC para meses é 99.
    - `ESTCIVMAE` e `ESCMAE2010` **(9.0 com ~0.2%)**: São sentinelas. Estão no final da lista, com frequências baixas, e o dicionário do SINASC reserva o 9 para "Ignorado" em variáveis categóricas codificadas de 1 a 5.
    - `SEXO` **(0 com 0.01%)**: O 0 aqui é sentinela para "Ignorado".

2. Onde o 9 é um VALOR REAL (dado válido)
    - Nestas colunas, o 9 faz parte de uma distribuição que vai diminuindo organicamente.
        - `QTDGESTANT, QTDPARTNOR, QTDFILVIVO`: * Observe a sequência: ...7, 8, 9, 10, 11...
            - O valor 9.0 aparece com uma frequência de **0.000650** (muito pequena). Se fosse um erro do sistema para "ignorado", esse número seria muito mais alto (como os 2% do `MESPRENAT`).
            - Aqui, 9 significa que a mãe realmente teve 9 gestações/partos. É um dado raro, mas biologicamente possível e importante para o modelo (multiparidade é fator de risco).
     
3. O caso especial: `RACACORMAE`
    - Note que em `RACACORMAE` apareceu um valor vazio com **2.5%**.
        - Isso é um "missing" disfarçado de string vazia. Você deve converter isso para NaN antes de qualquer outra coisa.


In [6]:
# Cria uma cópia para limpeza
df_clean = df.copy()

# 1. Padronização de tipos para fazer o mapeamento das sentinelas pois o .replace tem diferença entre types
cols_to_numeric = ['ESTCIVMAE', 'KOTELCHUCK', 'ESCMAE2010', 'GRAVIDEZ', 'SEXO', 'MESPRENAT', 'CONSPRENAT']

for col in cols_to_numeric:
    if col in df_clean.columns:
        # errors='coerce' limpa strings vazias ou caracteres estranhos
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 2. Mapeamento cirúrgico de sentinelas
sentinels_mapping = {
    'ESTCIVMAE': [9, 9.0],
    'KOTELCHUCK': [9, 9.0],
    'ESCMAE2010': [9, 9.0],
    'GRAVIDEZ': [9, 9.0],
    'SEXO': [0, 9, 9.0],
    'MESPRENAT': [99, 99.0],
    'CONSPRENAT': [99, 99.0]
}

for col, sentinels in sentinels_mapping.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace(sentinels, np.nan)


## Limpeza de Strings e Tipagem
Algumas strings na base que podem conter sugeira, como por exemplo `"MG"` tendo valores como `"MG "`(contendoespaços) o que é considerado diferente para o computador.

Objetivo: Remover espaços em branco e garantir que as colunas numéricas sejam de fato `float` ou `int`.

##### ps: a parte de tipagem já foi tratada no 01_eda.ipynb

In [7]:
# Tratamento de string vazia (ex: RACACORMAE)
df_clean = df_clean.replace(r'^\s*$', np.nan, regex=True)

## Restrição de Escopo: Gestações Únicas

Gestações gemelares (~60% de prematuridade) e trigemelares (>90%) têm etiologia distinta da prematuridade em gestações únicas (~10%): corionicidade, STFF e crescimento discordante determinam o desfecho, não os fatores socioeconômicos e obstétricos capturados pelo SINASC (Blickstein & Keith, 2005). O modelo restringe-se a **gestações únicas (`GRAVIDEZ == 1`)** — a prematuridade múltipla é estrutural e fora do escopo preditivo das features disponíveis.

In [8]:
before_scope = len(df_clean)
df_clean = df_clean[df_clean['GRAVIDEZ'] == 1.0].copy()
print(f"Excluídos (gemelar/trigemelar): {before_scope - len(df_clean):,}")
print(f"Mantidos (gestação única):      {len(df_clean):,}")

Excluídos (gemelar/trigemelar): 17,623
Mantidos (gestação única):      700,309


## Identificação e Remoção de Duplicadas

Após a "normalização" dos dados (sentinelas viraram nulos e espaços foram removidos) conseguimos uma base melhor tratada para identificar duplicadas e removê-las.

Objetivo: Identificar e remover as duplicadas

In [9]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Registros removidos (duplicatas): {before - len(df_clean):,}")
print(f"Registros finais: {len(df_clean):,}")

Registros removidos (duplicatas): 9,878
Registros finais: 690,431


## Identificação de Nulos

Agora que removemos as duplicadas precisamos tratar os nulos, isto é, nosso modelo não conseguiria processar valores não numéricos.

In [10]:
# 1. Ver contagem absoluta e percentual de nulos por coluna
null_summary = pd.DataFrame({
    'Nulos': df_clean.isnull().sum(),
    'Percentual (%)': (df_clean.isnull().mean() * 100).round(2)
})

# 2. Filtrar apenas as colunas que possuem algum nulo
print(null_summary[null_summary['Nulos'] > 0].sort_values(by='Nulos', ascending=False))

             Nulos  Percentual (%)
IDADEPAI    377650           54.70
MESPRENAT    23786            3.45
KOTELCHUCK   23441            3.40
RACACORMAE   16354            2.37
QTDPARTCES   10199            1.48
QTDPARTNOR   10075            1.46
QTDFILMORT    9852            1.43
QTDGESTANT    8583            1.24
QTDFILVIVO    7435            1.08
CONSPRENAT    7285            1.06
ESCMAE2010    5527            0.80
ESTCIVMAE     5319            0.77
SEXO            80            0.01
LATITUDE         6            0.00
LONGITUDE        6            0.00


## Feature Engineering

#### Mas porquê fazer a feature engineering _antes_ da imputação?
1. Preservação do Sinal (O caso do `PAI_AUSENTE`)
        Se você tem 50% de nulos na `IDADEPAI`, isso provavelmente indica que o pai não foi declarado no registro (de acordo com a realidade estatística de ausência paterna no Brasi).
        - **Se você imputar primeiro**: O nulo vira 30 (mediana). Quando você for criar a feature `PAI_AUSENTE`, não haverá mais nulos para identificar. O modelo verá um pai de 30 anos "comum".
        - **Se você fizer a engenharia antes**: Você marca `PAI_AUSENTE = 1`. Depois, você preenche a idade com `30`. O modelo agora tem duas informações: "Esse pai não está presente" **E** "Para fins de cálculo, considere a idade média de 30".

2. Evitar o "Ruído" em Regras de Negócio
        Algumas variáveis que criamos são baseadas em limites (ex: `PN_TARDIO` para quem começou o pré-natal após o 3º mês).
        - Se a mediana de `MESPRENAT` for `2` e você preencher os nulos antes, todas as mulheres que não informaram o mês de início serão classificadas como "Pré-natal Precoce" (mês 2).
        - Ao fazer a engenharia antes, você pode decidir se o nulo deve ser uma categoria à parte ou se deve ser tratado como risco, sem deixar que a mediana tome essa decisão por você.

3. Evitar Dados "Alucinados" (O caso da `PRIMIPARA`)
    Para saber se a mãe é de primeira viagem, olhamos se `QTDFILVIVO + QTDFILMORT == 0`.
    - Se você imputar a mediana em uma dessas colunas antes (digamos que a mediana de filhos vivos seja `1`), uma mãe que era nula (e possivelmente primípara) passará a ter `1` filho no cálculo. Você "inventou" um filho para ela e agora a feature `PRIMIPARA` será `0` (falso), quando poderia ser `1` (verdadeiro).


Se você inverte a ordem, você dá ao algoritmo dados "médios" e retira dele a capacidade de entender as exceções e os riscos sociais que o dado faltante representa.


#### Por que criar Ranges (Bins)?
Mães muito jovens ou em idade avançada possuem riscos biológicos e sociais completamente diferentes. Criar "faixas" ajuda o modelo a captar esses saltos de risco que um número contínuo (15, 16, 17...) às vezes mascara.

Aqui estão os ranges sugeridos baseados em literatura médica e nos padrões do SUS:

- **Extremo Baixa Idade (< 15 anos)**: Risco altíssimo, geralmente associado a vulnerabilidade social e imaturidade biológica.
- **Adolescente (15-19 anos)** Risco moderado/alto.
- **Idade Ideal (20-34 anos)**: Faixa de menor risco biológico (nossa base de comparação).
- **Idade Materna Avançada (35-44 anos)**: Risco crescente de hipertensão, diabetes gestacional e prematuridade.
- **Extremo Alta Idade (45+ anos)**: Risco muito elevado.

In [11]:
# Pré-natal Tardio (Inicio após o 3° mês)
df_clean['PNTARDIO'] = np.where(
    df_clean["MESPRENAT"].isna(), -1, (df_clean["MESPRENAT"] > 3).astype(int)
)

# Histórico de Perda (Se já teve algum filho morto)
df_clean['HISTPERDAFETAL'] = np.where(
    df_clean["QTDFILMORT"].isna(), -1, (df_clean['QTDFILMORT'] > 0).astype(int)
 )

# Primipara (Mãe na primeira gestação)
hist_children_absent = df_clean[["QTDFILVIVO", "QTDFILMORT"]].isna().any(axis=1)
total_children = df_clean[["QTDFILVIVO", "QTDFILMORT"]].sum(axis=1)

df_clean['PRIMIPARA'] = np.where(
    hist_children_absent, -1, (total_children == 0).astype(int)
)

# Categorização de idade
bins = [0, 14, 19, 34, 44 ,100]
labels = ['EXTBAIXA', 'ADOLESC', 'IDEAL', 'AVANCADA', 'EXTRALTA']

df_clean['FAIXAETAMAE'] = pd.cut(df_clean['IDADEMAE'], bins=bins, labels=labels)

# Flags de ausência para KOTELCHUCK e ESCMAE2010.
# Criadas antes da imputação para preservar o sinal "dado não informado"
# mesmo após o fillna com moda.
df_clean['KOTELCHUCK_IGNORADO'] = df_clean['KOTELCHUCK'].isna().astype(int)
df_clean['ESCMAE2010_IGNORADO'] = df_clean['ESCMAE2010'].isna().astype(int)

df_clean

,SEMAGESTAC,GESTACAO,IDADEMAE,ESCMAE2010,ESTCIVMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,GRAVIDEZ,MESPRENAT,CONSPRENAT,SEXO,KOTELCHUCK,IDADEPAI,LATITUDE,LONGITUDE,PREMATURO,PAI_AUSENTE,IDADEPAI_INVALIDA,PNTARDIO,HISTPERDAFETAL,PRIMIPARA,FAIXAETAMAE,KOTELCHUCK_IGNORADO,ESCMAE2010_IGNORADO
0,39.0,5.0,29,5.0,1.0,1,0.0,0.0,0.0,0.0,0.0,1.0,3.0,13.0,1.0,5.0,NaN,-21.6088,-45.5691,0,1,0,0,0,1,IDEAL,0,0
1,38.0,5.0,43,5.0,2.0,1,4.0,0.0,2.0,2.0,2.0,1.0,3.0,8.0,1.0,5.0,NaN,-21.8360,-45.4004,0,1,0,0,1,0,AVANCADA,0,0
2,39.0,5.0,33,5.0,1.0,1,1.0,0.0,1.0,1.0,0.0,1.0,2.0,11.0,1.0,5.0,NaN,-21.5556,-45.4364,0,1,0,0,0,0,IDEAL,0,0
3,38.0,5.0,26,2.0,1.0,2,4.0,0.0,4.0,4.0,0.0,1.0,5.0,4.0,2.0,2.0,NaN,-21.5556,-45.4364,0,1,0,1,0,0,IDEAL,0,0
4,37.0,5.0,20,4.0,1.0,1,0.0,0.0,0.0,0.0,0.0,1.0,4.0,7.0,1.0,2.0,NaN,-21.7579,-45.5391,0,1,0,1,0,1,IDEAL,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
717927,35.0,4.0,37,3.0,2.0,4,2.0,1.0,1.0,2.0,0.0,1.0,2.0,7.0,2.0,5.0,35.0,-19.9102,-43.9266,1,0,0,0,0,0,AVANCADA,0,0
717928,38.0,5.0,33,3.0,1.0,2,1.0,1.0,0.0,1.0,0.0,1.0,4.0,11.0,2.0,2.0,23.0,-19.9102,-43.9266,0,0,0,1,0,0,IDEAL,0,0
717929,38.0,5.0,30,3.0,2.0,1,1.0,1.0,0.0,1.0,0.0,1.0,2.0,8.0,2.0,5.0,32.0,-19.9102,-43.9266,0,0,0,0,0,0,IDEAL,0,0
717930,40.0,5.0,28,3.0,1.0,4,0.0,0.0,0.0,0.0,0.0,1.0,2.0,10.0,2.0,5.0,39.0,-19.9102,-43.9266,0,0,0,0,0,1,IDEAL,0,0


## Imputação - Tratamento de Nulos
Agora é hora de converter os valores nulos para que o modelo consiga interpretar todos os dados da tabela.

1. **Conversão categórica**: Já os nulos de colunas categóricas converteremos de para `-1` ou `IGNORADO` nos casos das colunas criadas pelo `pd.cut`
   - Isso vai facilitar na hora de fazermos o encoding

#### Por quê não fazer a conversão numérica agora?
 **Conversão numérica**: Os nulos de colunas numéricas converteremos para `mediana`, pois essa medida lida com outliers.
 Porém, essa etapa deve ser feita **SOMENTE após a separação da base de treino e base de teste**. Isso por quê quando ao trocarmos os dados por mediana ele irá considerar a mediana de 700k pessoas. Quando formos fazer o teste, o modelo já irá saber a média do banco de teste, provocando o **Data laeakage**.

 Sendo assim, seguiremos a seguinte ordem:
 1. iremos fazer a separação do banco entre treino e teste
 2. calculamos a mediana da base de treino
 3. aplicamos o valor da mediana da base de treino para a base de teste

### Decisões de escopo para imputação e encoding

**`IDADEPAI` — removida das features contínuas**
`IDADEPAI` possui ~54% de nulos. Com mais da metade do dataset sem valor, a imputação por mediana introduziria um ruído expressivo. O sinal clínico relevante já está capturado pela flag `PAI_AUSENTE` (pai não declarado no registro), que proxy para ausência paterna e é mantida como feature binária.

`IDADEPAI_INVALIDA` também é removida — são apenas 24 casos em 700k, sem poder preditivo.

**`GRAVIDEZ` — removida de `categoric_cols`**
Após o filtro para `GRAVIDEZ == 1.0`, todas as linhas têm o mesmo valor. Incluir essa coluna no encoding geraria dummies constantes, sem nenhum sinal para o modelo.

In [12]:
numeric_cols = [
    'IDADEMAE', 'QTDFILVIVO', 'QTDFILMORT', 'MESPRENAT',
    'QTDGESTANT', 'QTDPARTNOR', 'QTDPARTCES', 'LATITUDE', 'LONGITUDE',
]

categoric_cols = ['RACACORMAE', 'SEXO', 'FAIXAETAMAE', 'ESTCIVMAE', 'KOTELCHUCK', 'ESCMAE2010']

# Categorias sem ordem natural: NaN → -1 (IGNORADO como categoria explícita)
for col in categoric_cols:
    if col in df_clean.columns and col not in ('KOTELCHUCK', 'ESCMAE2010'):
        if df_clean[col].dtype.name == 'category':
            df_clean[col] = df_clean[col].cat.add_categories('IGNORADO').fillna('IGNORADO')
        else:
            df_clean[col] = df_clean[col].fillna(-1)

# KOTELCHUCK e ESCMAE2010 têm ordem natural — imputar com moda para manter
# o ordinal em escala limpa (0–4 e 0–5). A flag de ausência já foi criada acima.
for col in ('KOTELCHUCK', 'ESCMAE2010'):
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(df_clean.isnull().sum().sum(), "nulos restantes.")

454877 nulos restantes.


#### Dicionário de Mapeamento (Humanização)

Algumas colunas do dataset já vem com valores numéricos, ao aplicar o `One-Hot Enconding` ele transforma o nome dessas colunas de acordo com os valores que encontra na mesma. O que dificulta a interpretabilidade, por isso, iremos "traduzir" os nomes das colunas utilizando o [dicionário disponibilizado](https://diaad.s3.sa-east-1.amazonaws.com/sinasc/SINASC+-+Estrutura.pdf) pelo `OpenDataSus`


In [13]:
# Selecionando as categorias que farã o One-Hot
one_hot_cols = categoric_cols.copy()

# Verificando os valores contidos nessas colunas
for col in one_hot_cols:
    print(f'{col} : {df_clean[col].unique()}')

RACACORMAE : ['1' '2' '4' '3' -1 '5']
SEXO : [ 1.  2. -1.]
FAIXAETAMAE : ['IDEAL', 'AVANCADA', 'ADOLESC', 'EXTBAIXA', 'EXTRALTA']
Categories (6, object): ['EXTBAIXA' < 'ADOLESC' < 'IDEAL' < 'AVANCADA' < 'EXTRALTA' < 'IGNORADO']
ESTCIVMAE : [ 1.  2.  4.  5. -1.  3.]
KOTELCHUCK : [5. 2. 1. 4. 3.]
ESCMAE2010 : [5. 2. 4. 1. 3. 0.]


In [14]:
# Converter colunas em inteiros 
one_hot_int_cols = one_hot_cols.copy()

# Removendo colunas que não precisarão da humanização
one_hot_int_cols.remove('FAIXAETAMAE')

# Transformando em inteiros
for col in one_hot_int_cols:
    if col in df_clean.columns:
        # errors='coerce' transforma sujeira em NaN, fillna(-1) garante o inteiro
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(-1).astype(int)

# Verificando a transformação
for col in one_hot_cols:
    print(f'{col} : {df_clean[col].unique()}')

RACACORMAE : [ 1  2  4  3 -1  5]
SEXO : [ 1  2 -1]
FAIXAETAMAE : ['IDEAL', 'AVANCADA', 'ADOLESC', 'EXTBAIXA', 'EXTRALTA']
Categories (6, object): ['EXTBAIXA' < 'ADOLESC' < 'IDEAL' < 'AVANCADA' < 'EXTRALTA' < 'IGNORADO']
ESTCIVMAE : [ 1  2  4  5 -1  3]
KOTELCHUCK : [5 2 1 4 3]
ESCMAE2010 : [5 2 4 1 3 0]


In [15]:
# KOTELCHUCK e ESCMAE2010 já foram imputados com moda — sem NaN, sem -1.
# ESCMAE2010: valores 0–5 já são o próprio ordinal (sem escolaridade → superior completo).
df_clean['ESCMAE2010_ORDINAL'] = df_clean['ESCMAE2010'].astype(int)

# KOTELCHUCK: valores 1–5, shift de -1 para alinhar o ordinal em 0–4.
df_clean['KOTELCHUCK_ORDINAL'] = (df_clean['KOTELCHUCK'] - 1).astype(int)

In [16]:
sinasc_dictionary = {
    'SEXO': {1: 'MASC', 2: 'FEM', -1: 'IGNORADO'},
    'RACACORMAE': {1: 'BRANCA', 2: 'PRETA', 3: 'AMARELA', 4: 'PARDA', 5: 'INDIGENA', -1: 'IGNORADO'},
    'ESTCIVMAE': {1: 'SOLTEIRA', 2: 'CASADA', 3: 'VIUVA', 4: 'DIVORCIADA', 5: 'UNIAO_ESTAVEL', -1: 'IGNORADO'},
    'GRAVIDEZ': {1: 'UNICA', 2: 'DUPLA', 3: 'TRIPMAIS', -1: 'IGNORADO'},
    "KOTELCHUCK": {
        1: "SEM_PRENATAL",
        2: "INADEQUADO",
        3: "INTERMEDIARIO",
        4: "ADEQUADO",
        5: "MAIS_QUE_ADEQUADO",
        -1: "IGNORADO"
    },
    "ESCMAE2010": {
        0: "SEM_ESCOLARIDADE",
        1: "FUNDAMENTAL_I",
        2: "FUNDAMENTAL_II",
        3: "MEDIO",
        4: "SUPERIOR_INCOMPLETO", 
        5: "SUPERIOR_COMPLETO",
        -1: "IGNORADO"
    }
}

for col, value in sinasc_dictionary.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].map(value).fillna('IGNORADO')     

df_clean

,SEMAGESTAC,GESTACAO,IDADEMAE,ESCMAE2010,ESTCIVMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,GRAVIDEZ,MESPRENAT,CONSPRENAT,SEXO,KOTELCHUCK,IDADEPAI,LATITUDE,LONGITUDE,PREMATURO,PAI_AUSENTE,IDADEPAI_INVALIDA,PNTARDIO,HISTPERDAFETAL,PRIMIPARA,FAIXAETAMAE,KOTELCHUCK_IGNORADO,ESCMAE2010_IGNORADO,ESCMAE2010_ORDINAL,KOTELCHUCK_ORDINAL
0,39.0,5.0,29,SUPERIOR_COMPLETO,SOLTEIRA,BRANCA,0.0,0.0,0.0,0.0,0.0,UNICA,3.0,13.0,MASC,MAIS_QUE_ADEQUADO,NaN,-21.6088,-45.5691,0,1,0,0,0,1,IDEAL,0,0,5,4
1,38.0,5.0,43,SUPERIOR_COMPLETO,CASADA,BRANCA,4.0,0.0,2.0,2.0,2.0,UNICA,3.0,8.0,MASC,MAIS_QUE_ADEQUADO,NaN,-21.8360,-45.4004,0,1,0,0,1,0,AVANCADA,0,0,5,4
2,39.0,5.0,33,SUPERIOR_COMPLETO,SOLTEIRA,BRANCA,1.0,0.0,1.0,1.0,0.0,UNICA,2.0,11.0,MASC,MAIS_QUE_ADEQUADO,NaN,-21.5556,-45.4364,0,1,0,0,0,0,IDEAL,0,0,5,4
3,38.0,5.0,26,FUNDAMENTAL_II,SOLTEIRA,PRETA,4.0,0.0,4.0,4.0,0.0,UNICA,5.0,4.0,FEM,INADEQUADO,NaN,-21.5556,-45.4364,0,1,0,1,0,0,IDEAL,0,0,2,1
4,37.0,5.0,20,SUPERIOR_INCOMPLETO,SOLTEIRA,BRANCA,0.0,0.0,0.0,0.0,0.0,UNICA,4.0,7.0,MASC,INADEQUADO,NaN,-21.7579,-45.5391,0,1,0,1,0,1,IDEAL,0,0,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
717927,35.0,4.0,37,MEDIO,CASADA,PARDA,2.0,1.0,1.0,2.0,0.0,UNICA,2.0,7.0,FEM,MAIS_QUE_ADEQUADO,35.0,-19.9102,-43.9266,1,0,0,0,0,0,AVANCADA,0,0,3,4
717928,38.0,5.0,33,MEDIO,SOLTEIRA,PRETA,1.0,1.0,0.0,1.0,0.0,UNICA,4.0,11.0,FEM,INADEQUADO,23.0,-19.9102,-43.9266,0,0,0,1,0,0,IDEAL,0,0,3,1
717929,38.0,5.0,30,MEDIO,CASADA,BRANCA,1.0,1.0,0.0,1.0,0.0,UNICA,2.0,8.0,FEM,MAIS_QUE_ADEQUADO,32.0,-19.9102,-43.9266,0,0,0,0,0,0,IDEAL,0,0,3,4
717930,40.0,5.0,28,MEDIO,SOLTEIRA,PARDA,0.0,0.0,0.0,0.0,0.0,UNICA,2.0,10.0,FEM,MAIS_QUE_ADEQUADO,39.0,-19.9102,-43.9266,0,0,0,0,0,1,IDEAL,0,0,3,4


### Preparação e Treino (Split)

Nesta etapa, pegamos o seu dataset original e o dividimos em dois grupos distintos: **Treino** (geralmente 80%) e **Teste** (20%).

- **O que acontece:** O grupo de treino é o "livro" que o modelo vai ler para aprender. O grupo de teste é a "prova final" com questões que ele nunca viu antes.
- **A "Estratificação":** Como você tem poucos prematuros (11%), usamos o `stratify=y` para garantir que tanto o treino quanto o teste tenham exatamente esses mesmos 11%. Sem isso, você poderia acabar com um teste sem nenhum prematuro para validar!


In [17]:
from sklearn.model_selection import train_test_split

In [18]:
y = df_clean['PREMATURO']

cols_to_drop = ['SEMAGESTAC', 'GESTACAO', 'PREMATURO', 'CONSPRENAT', 'GRAVIDEZ', 'IDADEPAI', 'IDADEPAI_INVALIDA']
X = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [19]:
y_train.value_counts()

PREMATURO
0    495998
1     56346
Name: count, dtype: int64

### Estratégia de imputação: mediana vs moda

A maioria das colunas numéricas recebe a **mediana** do treino — medida robusta a outliers, calculada apenas no treino para evitar data leakage.

**Exceção: `QTDFILVIVO` e `QTDGESTANT` recebem a moda (= 0)**

Para variáveis de contagem com distribuição concentrada em zero, mediana e moda divergem:

| Coluna | Mediana | Moda |
|--------|---------|------|
| `QTDFILVIVO` | 1.0 | **0.0** |
| `QTDGESTANT` | 1.0 | **0.0** |

Imputar a mediana (1) nessas colunas equivale a "inventar" um filho ou uma gestação anterior para quem não informou. Isso é especialmente problemático porque `PRIMIPARA` é derivada de `QTDFILVIVO` — uma imputação de 1 forçaria `PRIMIPARA = 0` de forma artificial para registros com dado ausente.

A moda (0) é a escolha mais conservadora: assume ausência de histórico quando a informação não está disponível.

In [20]:
numeric_cols_model = [col for col in numeric_cols if col in X_train.columns]

# Colunas de contagem onde a moda (0) é mais conservadora que a mediana (1):
# imputar mediana=1 "inventaria" filho ou gestação anterior para quem não informou.
mode_cols = ['QTDFILVIVO', 'QTDGESTANT']

for col in numeric_cols_model:
    if col in mode_cols:
        train_ref = X_train[col].mode()[0]
    else:
        train_ref = X_train[col].median()

    X_train[col] = X_train[col].fillna(train_ref)
    X_test[col] = X_test[col].fillna(train_ref)

## Enconding
Nesta etapa, convertemos colunas categóricas em numéricas para que o modelo possa processá-las matematicamente, seguindo dois critérios:

- **Diferenciação de Pesos**: Para variáveis sem hierarquia (como `SEXO` ou `RACACORMAE`), utilizamos o **One-Hot Encoding**. Isso evita que o modelo atribua pesos errados (ex: interpretar que **`2 > 1`**) a categorias puramente nominais.
- **Estratégia de Integridade (Anti-Leakage)**: Realizamos o cálculo de imputação (como a **mediana**) apenas após a separação entre treino e teste.

#### Por que separar antes?
Para garantir que a estatística do conjunto de teste não "vaze" para o treino. Em produção, o modelo usará a "régua" aprendida no passado (treino) para lidar com novos dados, garantindo uma avaliação real de sua performance.

### Por que `drop_first=False`?

Em One-Hot Encoding, `drop_first=True` remove uma categoria de referência para evitar multicolinearidade perfeita — necessário para regressão linear sem regularização.

Neste projeto usamos `drop_first=False` por dois motivos:

1. **Interpretabilidade**: cada categoria tem seu coeficiente explícito. Não é necessário inferir o efeito da categoria de referência por subtração.
2. **Regularização controla multicolinearidade**: todos os modelos testados (LogReg com L1/L2, RandomForest, HistGB) lidam com colunas correlacionadas sem instabilidade numérica.

A base "rica" mantém todas as dummies. Os cenários de modelagem (`pipeline.py`) selecionam quais representações usar — dummies, ordinais ou ambas.

In [21]:
X_train_encoded = pd.get_dummies(X_train, columns=one_hot_cols, drop_first=False)
X_test_encoded = pd.get_dummies(X_test, columns=one_hot_cols, drop_first=False)

X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

bool_cols = X_train_encoded.select_dtypes(include=['bool']).columns
X_train_encoded[bool_cols] = X_train_encoded[bool_cols].astype(int)
X_test_encoded[bool_cols] = X_test_encoded[bool_cols].astype(int)

In [22]:
X_train_encoded.head(10)

,IDADEMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,MESPRENAT,LATITUDE,LONGITUDE,PAI_AUSENTE,PNTARDIO,HISTPERDAFETAL,PRIMIPARA,KOTELCHUCK_IGNORADO,ESCMAE2010_IGNORADO,ESCMAE2010_ORDINAL,KOTELCHUCK_ORDINAL,RACACORMAE_AMARELA,RACACORMAE_BRANCA,RACACORMAE_IGNORADO,RACACORMAE_INDIGENA,RACACORMAE_PARDA,RACACORMAE_PRETA,SEXO_FEM,SEXO_IGNORADO,SEXO_MASC,FAIXAETAMAE_EXTBAIXA,FAIXAETAMAE_ADOLESC,FAIXAETAMAE_IDEAL,FAIXAETAMAE_AVANCADA,FAIXAETAMAE_EXTRALTA,FAIXAETAMAE_IGNORADO,ESTCIVMAE_CASADA,ESTCIVMAE_DIVORCIADA,ESTCIVMAE_IGNORADO,ESTCIVMAE_SOLTEIRA,ESTCIVMAE_UNIAO_ESTAVEL,ESTCIVMAE_VIUVA,KOTELCHUCK_ADEQUADO,KOTELCHUCK_INADEQUADO,KOTELCHUCK_INTERMEDIARIO,KOTELCHUCK_MAIS_QUE_ADEQUADO,KOTELCHUCK_SEM_PRENATAL,ESCMAE2010_FUNDAMENTAL_I,ESCMAE2010_FUNDAMENTAL_II,ESCMAE2010_MEDIO,ESCMAE2010_SEM_ESCOLARIDADE,ESCMAE2010_SUPERIOR_COMPLETO,ESCMAE2010_SUPERIOR_INCOMPLETO
208361,29,2.0,0.0,2.0,2.0,0.0,1.0,-22.2266,-45.9389,0,0,0,0,0,0,4,4,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1
151790,22,2.0,2.0,0.0,2.0,0.0,2.0,-15.4802,-44.3639,1,-1,0,0,1,0,2,4,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0
58576,26,1.0,0.0,1.0,1.0,0.0,8.0,-22.2859,-44.8680,0,1,0,0,0,0,5,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0
124394,14,0.0,0.0,0.0,0.0,0.0,3.0,-17.6888,-42.5147,1,0,0,1,0,0,2,4,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0
496103,22,1.0,1.0,0.0,1.0,0.0,2.0,-18.8690,-48.8810,0,0,0,0,0,0,2,4,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0
111651,27,0.0,0.0,0.0,0.0,0.0,2.0,-19.6308,-44.0383,1,0,0,1,0,0,3,4,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
147862,29,1.0,1.0,0.0,1.0,0.0,2.0,-15.1514,-42.8718,1,0,0,0,0,0,3,4,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
256428,25,0.0,0.0,0.0,0.0,0.0,2.0,-20.1446,-44.8912,0,0,0,1,0,0,3,4,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0
407472,30,0.0,0.0,0.0,0.0,0.0,1.0,-20.1446,-44.8912,1,0,0,1,0,0,4,4,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1
407447,32,0.0,0.0,0.0,0.0,0.0,1.0,-19.8713,-44.9847,0,0,0,1,0,0,3,4,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0


In [23]:
X_test_encoded.head(10)

,IDADEMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,MESPRENAT,LATITUDE,LONGITUDE,PAI_AUSENTE,PNTARDIO,HISTPERDAFETAL,PRIMIPARA,KOTELCHUCK_IGNORADO,ESCMAE2010_IGNORADO,ESCMAE2010_ORDINAL,KOTELCHUCK_ORDINAL,RACACORMAE_AMARELA,RACACORMAE_BRANCA,RACACORMAE_IGNORADO,RACACORMAE_INDIGENA,RACACORMAE_PARDA,RACACORMAE_PRETA,SEXO_FEM,SEXO_IGNORADO,SEXO_MASC,FAIXAETAMAE_EXTBAIXA,FAIXAETAMAE_ADOLESC,FAIXAETAMAE_IDEAL,FAIXAETAMAE_AVANCADA,FAIXAETAMAE_EXTRALTA,FAIXAETAMAE_IGNORADO,ESTCIVMAE_CASADA,ESTCIVMAE_DIVORCIADA,ESTCIVMAE_IGNORADO,ESTCIVMAE_SOLTEIRA,ESTCIVMAE_UNIAO_ESTAVEL,ESTCIVMAE_VIUVA,KOTELCHUCK_ADEQUADO,KOTELCHUCK_INADEQUADO,KOTELCHUCK_INTERMEDIARIO,KOTELCHUCK_MAIS_QUE_ADEQUADO,KOTELCHUCK_SEM_PRENATAL,ESCMAE2010_FUNDAMENTAL_I,ESCMAE2010_FUNDAMENTAL_II,ESCMAE2010_MEDIO,ESCMAE2010_SEM_ESCOLARIDADE,ESCMAE2010_SUPERIOR_COMPLETO,ESCMAE2010_SUPERIOR_INCOMPLETO
172519,43,0.0,0.0,0.0,0.0,0.0,1.0,-22.3136,-46.3264,0,0,-1,-1,1,0,3,4,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
214233,27,0.0,0.0,0.0,0.0,0.0,2.0,-20.9167,-46.9837,1,0,0,1,0,0,5,4,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
424765,21,1.0,1.0,0.0,1.0,0.0,2.0,-21.5556,-45.4364,1,-1,0,0,0,0,3,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0
458543,22,1.0,1.0,0.0,1.0,0.0,1.0,-15.6032,-44.3910,1,0,0,0,1,0,3,4,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0
302650,36,2.0,1.0,1.0,2.0,0.0,3.0,-18.9772,-49.4639,1,0,0,0,0,0,2,3,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0
166378,25,2.0,2.0,0.0,2.0,0.0,3.0,-16.7282,-43.8578,1,0,0,0,0,0,3,4,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0
267763,43,3.0,0.0,2.0,2.0,1.0,1.0,-21.5556,-45.4364,1,0,1,0,0,0,3,4,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0
389746,40,7.0,6.0,1.0,7.0,0.0,2.0,-19.9794,-44.4318,1,0,0,0,0,0,3,4,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0
11336,42,1.0,1.0,0.0,1.0,0.0,1.0,-19.9321,-44.0539,0,0,0,0,0,0,5,4,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
419390,21,2.0,0.0,2.0,2.0,0.0,2.0,-18.9141,-48.2749,1,0,0,0,0,0,3,4,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0


In [24]:
X_train_encoded.shape

(552344, 49)

#### A quantidade de colunas AUMENTOU e agora? 

Será que mesmo que a quantidade de colunas tenha aumentado vale a pena considerar uma prática de dimensionamento? Por quê? 

A resposta é **não**.

##### Motivos:
- Contém colunas essenciais
- O tamanho do dataset é considerado saudável para modelos atuais.
  - A diferença de 21 para 46 colunas para modelos atuais é irrelevante pois mesmo que tenha dobrado, ainda temos apenas 700k de linhas. O que é uma quantidade boa para faze um modelo legal, mas não é uma quantidade absurda.
- Se aplicarmos um **PCA** aqui iriamos perder a relevância clínica (que é a ALMA do nosso dataset)


### Padronização (Scaling)

Aqui transformamos todas as variáveis para que tenham a mesma escala (média 0 e desvio padrão 1).

- **O que acontece:** A **Latitude** (ex: -23) e o **Número de Consultas** (ex: 7) são números de escalas muito diferentes. O Scaler coloca todos em uma "régua" comum.
- **O detalhe técnico:** O modelo para de olhar para o valor absoluto e passa a olhar para o quanto aquele valor se desvia da média.

In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_encoded), columns=X_train_encoded.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_encoded), columns=X_test_encoded.columns)

#### Por que fazemos nessa ordem?
A ordem correta é **Split primeiro, Padronização depois**. O motivo é evitar o **Data Leakage** (Vazamento de Dados).

Se padronizarmos o dataset inteiro **antes** de dividir, o cálculo da média e do desvio padrão incluiria informações dos dados de teste. Isso significa que:

1. O grupo de treino "saberia" informações sobre a distribuição global dos dados que ele não deveria saber.
2. Isso cria um modelo "viciado" que performa muito bem no seu notebook, mas falha miseravelmente quando recebe dados reais de um hospital no futuro, pois ele aprendeu com base em médias que incluíam o "futuro" (o teste).

### Persistência dos artefatos

Salva `X_train_scaled`, `X_test_scaled`, `y_train` e `y_test` em `data/` para que o `03_modeling.ipynb` possa carregá-los sem re-executar todo o pipeline.

O índice de `y_train`/`y_test` é resetado para alinhar com o índice 0-based criado pelo `StandardScaler` no passo anterior.

In [26]:
from utils import save_if_different

y_train_reset = y_train.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

save_if_different(X_train_scaled, '../data/X_train.parquet')
save_if_different(X_test_scaled, '../data/X_test.parquet')
save_if_different(y_train_reset.to_frame(), '../data/y_train.parquet')
save_if_different(y_test_reset.to_frame(), '../data/y_test.parquet')

Arquivo em sua última versão!
Arquivo em sua última versão!
Arquivo em sua última versão!
Arquivo em sua última versão!
